# 03. CNN 기초 (Convolutional Neural Networks)

## 학습 목표
- Convolution 연산의 원리를 직접 구현하며 이해
- CNN 구성요소 (Conv2d, Pooling, Stride, Padding) 파악
- Feature Map을 시각화하여 필터가 감지하는 패턴 확인
- LeNet-5 아키텍처 구현
- CIFAR-10 이미지 분류 전체 파이프라인 구현

## 참고 자료
- [CS231n - Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/)
- [Yann LeCun - LeNet-5 (1998)](http://yann.lecun.com/exdb/lenet/)

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

## 1. Convolution 연산

Convolution은 **필터(커널)**를 이미지 위에서 슬라이딩하며 element-wise 곱의 합을 구하는 연산.

**왜 Convolution인가?**
- FC(Fully Connected)는 이미지의 모든 픽셀을 1D로 펼쳐서 연결 -> 공간 정보 손실, 파라미터 폭발
- CNN은 **지역적 패턴**을 감지하고, **파라미터를 공유**하여 효율적

### 1D Convolution

$$(f * g)[n] = \sum_{k} f[k] \cdot g[n-k]$$

In [ ]:
# 1D Convolution 직접 구현
def conv1d_manual(signal, kernel):
    """1D convolution (valid mode, no padding)"""
    n = len(signal)
    k = len(kernel)
    output_len = n - k + 1
    output = np.zeros(output_len)
    for i in range(output_len):
        output[i] = np.sum(signal[i:i+k] * kernel)
    return output

# 예시: 간단한 1D 신호에 edge detection 필터 적용
signal = np.array([0, 0, 0, 1, 1, 1, 1, 0, 0, 0], dtype=float)
edge_kernel = np.array([-1, 0, 1], dtype=float)  # 엣지 감지
smooth_kernel = np.array([1/3, 1/3, 1/3], dtype=float)  # 스무딩

edge_output = conv1d_manual(signal, edge_kernel)
smooth_output = conv1d_manual(signal, smooth_kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 3))

axes[0].stem(signal, basefmt='k-')
axes[0].set_title('Original Signal'); axes[0].grid(True, alpha=0.3)

axes[1].stem(edge_output, basefmt='k-', linefmt='r-', markerfmt='ro')
axes[1].set_title('Edge Detection [-1, 0, 1]'); axes[1].grid(True, alpha=0.3)

axes[2].stem(smooth_output, basefmt='k-', linefmt='g-', markerfmt='go')
axes[2].set_title('Smoothing [1/3, 1/3, 1/3]'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"원본 신호:     {signal}")
print(f"Edge 결과:     {edge_output}")
print(f"Smooth 결과:   {np.round(smooth_output, 3)}")

### 2D Convolution

이미지에 적용하는 2D convolution. 필터를 이미지 위에서 좌우, 상하로 슬라이딩.

$$(I * K)[i,j] = \sum_m \sum_n I[i+m, j+n] \cdot K[m, n]$$

출력 크기: $(H - K_h + 1) \times (W - K_w + 1)$ (valid, no padding)

In [ ]:
# 2D Convolution 직접 구현
def conv2d_manual(image, kernel):
    """2D convolution (valid mode, no padding)"""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = h - kh + 1
    out_w = w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# 테스트 이미지: 간단한 패턴
image = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
], dtype=float)

# 다양한 필터
kernels = {
    'Vertical Edge': np.array([[-1, 0, 1],
                                [-1, 0, 1],
                                [-1, 0, 1]], dtype=float),
    'Horizontal Edge': np.array([[-1, -1, -1],
                                  [0,  0,  0],
                                  [1,  1,  1]], dtype=float),
    'Sharpen': np.array([[0, -1,  0],
                          [-1,  5, -1],
                          [0, -1,  0]], dtype=float),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image'); axes[0].axis('off')

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    output = conv2d_manual(image, kernel)
    ax.imshow(output, cmap='RdBu_r')
    ax.set_title(name); ax.axis('off')

plt.tight_layout()
plt.show()

---
## 2. CNN 구성요소

### Conv2d

PyTorch의 `nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`

출력 크기 공식:
$$H_{out} = \frac{H_{in} - K + 2P}{S} + 1$$

- $K$: 커널 크기
- $P$: 패딩
- $S$: 스트라이드

### Pooling

- **Max Pooling**: 영역 내 최대값 선택 -> 주요 특징 보존, 위치 불변성
- **Average Pooling**: 영역 내 평균값 -> 부드러운 다운샘플링

In [ ]:
# Stride, Padding, Pooling 효과 시각화

# 예시 입력 (1 batch, 1 channel, 6x6)
x = torch.arange(36, dtype=torch.float32).reshape(1, 1, 6, 6)
print(f"입력 shape: {x.shape}")
print(f"입력:\n{x[0, 0]}")

# 다양한 설정의 Conv2d
configs = [
    ('kernel=3, stride=1, pad=0', nn.Conv2d(1, 1, 3, stride=1, padding=0)),
    ('kernel=3, stride=1, pad=1', nn.Conv2d(1, 1, 3, stride=1, padding=1)),
    ('kernel=3, stride=2, pad=0', nn.Conv2d(1, 1, 3, stride=2, padding=0)),
]

print("\n=== Conv2d 출력 크기 ===")
for name, conv in configs:
    out = conv(x)
    H_in = 6
    K = 3
    P = conv.padding[0]
    S = conv.stride[0]
    H_calc = (H_in - K + 2*P) // S + 1
    print(f"  {name:30s} -> output shape: {out.shape[2]}x{out.shape[3]}  (공식: ({H_in}-{K}+2*{P})/{S}+1 = {H_calc})")

# Pooling
print("\n=== Pooling ===")
x_pool = torch.tensor([[[[1, 3, 2, 4],
                          [5, 6, 1, 2],
                          [7, 8, 3, 0],
                          [4, 2, 9, 1]]]], dtype=torch.float32)
print(f"입력:\n{x_pool[0, 0]}")

max_pool = nn.MaxPool2d(2)
avg_pool = nn.AvgPool2d(2)
print(f"\nMax Pooling (2x2):\n{max_pool(x_pool)[0, 0]}")
print(f"\nAvg Pooling (2x2):\n{avg_pool(x_pool)[0, 0]}")

---
## 3. Feature Map 시각화

학습된 CNN 필터가 **무엇을 감지하는지** 시각화한다.

- 초기 레이어: 엣지, 색상 등 저수준 특징
- 깊은 레이어: 텍스처, 패턴, 물체 부분 등 고수준 특징

In [ ]:
# 사전 학습된 모델의 필터 시각화
# 간단한 CNN을 만들고 손으로 설정한 필터로 Feature Map 시각화

# 테스트 이미지 생성: 다양한 엣지를 포함하는 이미지
np.random.seed(42)
test_img = np.zeros((32, 32))
test_img[5:27, 5:27] = 1.0         # 사각형
test_img[10:22, 10:22] = 0.0        # 안쪽 비움
test_img[14:18, 14:18] = 1.0        # 중앙 작은 사각형

# 수동 필터 정의
manual_filters = {
    'Vertical Edge': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),  # Sobel X
    'Horizontal Edge': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),  # Sobel Y
    'Diagonal': np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]], dtype=np.float32),
    'Gaussian Blur': np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32) / 16,
}

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# 상단: 필터 자체
axes[0, 0].imshow(test_img, cmap='gray')
axes[0, 0].set_title('Input Image'); axes[0, 0].axis('off')

for idx, (name, kernel) in enumerate(manual_filters.items()):
    axes[0, idx+1].imshow(kernel, cmap='RdBu_r')
    axes[0, idx+1].set_title(f'Filter: {name}', fontsize=9)
    axes[0, idx+1].axis('off')

# 하단: Feature Map (convolution 결과)
axes[1, 0].imshow(test_img, cmap='gray')
axes[1, 0].set_title('Input Image'); axes[1, 0].axis('off')

for idx, (name, kernel) in enumerate(manual_filters.items()):
    # PyTorch Conv2d로 적용
    conv = nn.Conv2d(1, 1, 3, padding=1, bias=False)
    conv.weight.data = torch.tensor(kernel).reshape(1, 1, 3, 3)
    
    img_tensor = torch.tensor(test_img, dtype=torch.float32).reshape(1, 1, 32, 32)
    with torch.no_grad():
        feature_map = conv(img_tensor).squeeze().numpy()
    
    axes[1, idx+1].imshow(feature_map, cmap='RdBu_r')
    axes[1, idx+1].set_title(f'Feature Map: {name}', fontsize=9)
    axes[1, idx+1].axis('off')

plt.suptitle('Filters (top) and Feature Maps (bottom)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. CNN 아키텍처: LeNet-5

Yann LeCun이 1998년에 제안한 역사적인 CNN 아키텍처. 손글씨 숫자 인식에 사용.

### 구조
```
Input (1x32x32)
  -> Conv2d(1, 6, 5)    -> ReLU -> AvgPool2d(2)    # 6x14x14
  -> Conv2d(6, 16, 5)   -> ReLU -> AvgPool2d(2)    # 16x5x5
  -> Flatten                                        # 400
  -> Linear(400, 120)   -> ReLU                     # 120
  -> Linear(120, 84)    -> ReLU                     # 84
  -> Linear(84, 10)                                 # 10 (classes)
```

In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 구현"""
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature extractor
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, padding=2)   # 입력: 3채널 (RGB)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool = nn.AvgPool2d(2, 2)
        
        # Classifier
        self.fc1 = nn.Linear(16 * 6 * 6, 120)  # CIFAR-10 (32x32) 기준
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
    
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))   # (B, 6, 16, 16)
        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))   # (B, 16, 6, 6)
        # Flatten
        x = x.view(x.size(0), -1)              # (B, 576)
        # FC layers
        x = F.relu(self.fc1(x))                # (B, 120)
        x = F.relu(self.fc2(x))                # (B, 84)
        x = self.fc3(x)                         # (B, 10)
        return x

# 모델 구조 확인
model = LeNet5()
print(model)

# 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
print(f"\n총 파라미터 수: {total_params:,}")

# 테스트 forward
dummy_input = torch.randn(1, 3, 32, 32)
output = model(dummy_input)
print(f"입력 shape: {dummy_input.shape} -> 출력 shape: {output.shape}")

---
## 5. CIFAR-10 분류

전체 파이프라인: 데이터 로드 -> 모델 정의 -> 학습 -> 평가

CIFAR-10: 32x32 컬러 이미지, 10개 클래스 (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck)

In [ ]:
# 데이터 로드
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                           shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                          shuffle=False, num_workers=2)

classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"학습 데이터: {len(trainset)}개")
print(f"테스트 데이터: {len(testset)}개")
print(f"클래스: {classes}")

In [ ]:
# 데이터 샘플 시각화
def imshow(img, title=None):
    """정규화 해제 후 이미지 표시"""
    img = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1) + \
          torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    img = torch.clamp(img, 0, 1)
    npimg = img.numpy()
    return np.transpose(npimg, (1, 2, 0))

# 랜덤 샘플 표시
dataiter = iter(trainloader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imshow(images[i]))
    ax.set_title(classes[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 모델, 손실함수, 옵티마이저
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LeNet5(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습
num_epochs = 10
train_losses = []
train_accs = []
test_accs = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    train_loss = running_loss / len(trainloader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 테스트 정확도
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    test_acc = 100. * correct / total
    test_accs.append(test_acc)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

---
## 6. 학습 과정 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax = axes[0]
ax.plot(range(1, num_epochs+1), train_losses, 'b-o', label='Train Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss'); ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy curves
ax = axes[1]
ax.plot(range(1, num_epochs+1), train_accs, 'b-o', label='Train Acc')
ax.plot(range(1, num_epochs+1), test_accs, 'r-o', label='Test Acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Train vs Test Accuracy'); ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"최종 Test Accuracy: {test_accs[-1]:.2f}%")
print(f"참고: LeNet-5는 단순한 구조라 ~65% 정도가 일반적")
print(f"더 깊은 모델 (ResNet 등)은 90%+ 달성 가능")

In [ ]:
# 학습된 Conv1 필터 시각화
filters = model.conv1.weight.data.cpu()
print(f"Conv1 필터 shape: {filters.shape}  (out_channels, in_channels, H, W)")

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    # 3채널 필터를 RGB 이미지로 시각화
    f = filters[i].permute(1, 2, 0).numpy()
    f = (f - f.min()) / (f.max() - f.min())  # 0-1 정규화
    ax.imshow(f)
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.suptitle('Learned Conv1 Filters', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 클래스별 정확도
class_correct = [0] * 10
class_total = [0] * 10

model.eval()
with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        for i in range(len(targets)):
            label = targets[i].item()
            class_correct[label] += (predicted[i] == label).item()
            class_total[label] += 1

print("=== 클래스별 정확도 ===")
for i in range(10):
    acc = 100 * class_correct[i] / class_total[i]
    print(f"  {classes[i]:12s}: {acc:.1f}%")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 더 깊은 CNN 만들기

LeNet-5보다 더 깊은 CNN을 만들어 CIFAR-10 정확도를 높여보세요.

제안 구조:
```
Conv2d(3, 32, 3, padding=1) -> BN -> ReLU -> Conv2d(32, 32, 3, padding=1) -> BN -> ReLU -> MaxPool2d(2)
Conv2d(32, 64, 3, padding=1) -> BN -> ReLU -> Conv2d(64, 64, 3, padding=1) -> BN -> ReLU -> MaxPool2d(2)
Flatten -> Linear(64*8*8, 256) -> ReLU -> Dropout(0.5) -> Linear(256, 10)
```

In [ ]:
# TODO: 위 구조의 DeeperCNN 클래스를 구현하세요
# 힌트:
# 1. nn.BatchNorm2d(channels) 사용
# 2. nn.Dropout(0.5) 사용
# 3. 10 epoch 학습 후 LeNet-5와 정확도 비교


### 연습 2: Feature Map 시각화 함수 구현

임의의 CIFAR-10 이미지를 학습된 모델에 통과시키며, 각 Conv layer의 Feature Map을 시각화하는 함수를 만드세요.

In [ ]:
# TODO: Feature Map 시각화 함수 구현
# 힌트:
# 1. 모델의 conv1, conv2 레이어에 forward hook을 등록하여 중간 출력을 캡처
#    hook_fn = lambda module, input, output: feature_maps.append(output)
#    model.conv1.register_forward_hook(hook_fn)
# 2. 또는 모델의 forward를 단계별로 실행
#    x = F.relu(model.conv1(img_tensor))
#    # x의 각 채널을 시각화
# 3. 각 채널을 subplot으로 시각화


---
## 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| Convolution | 필터를 슬라이딩하며 지역 패턴 감지. 파라미터 공유로 효율적 |
| Stride | 필터 이동 간격. 크면 출력 크기 줄어듦 |
| Padding | 입력 테두리에 0 추가. 출력 크기 유지에 사용 |
| Pooling | 다운샘플링. Max Pool (주요 특징 보존), Avg Pool (부드럽게) |
| Feature Map | 필터 적용 결과. 초기는 엣지, 깊으면 복잡한 패턴 |
| LeNet-5 | 초기 CNN (1998). Conv -> Pool -> FC 구조의 원형 |
| BatchNorm | 학습 안정화, 더 높은 학습률 사용 가능 |
| Dropout | 과적합 방지. 학습 시 랜덤하게 뉴런 비활성화 |

**다음 노트북**: [04-rnn-and-sequence.ipynb](04-rnn-and-sequence.ipynb) - RNN과 시퀀스 모델링